# Preprocessing, Dataset 2: Simulated Fraud Detection (fraudTrain.csv + fraudTest.csv)

In [1]:
import pandas as pd
import numpy as np

TRAIN_PATH = '../Dataset/fraud-detection/fraudTrain.csv'
TEST_PATH = '../Dataset/fraud-detection/fraudTest.csv'
TRAIN_OUT = '../Dataset/processed/frauddetection_train_clean.csv'
TEST_OUT = '../Dataset/processed/frauddetection_test_clean.csv'

In [2]:
train = pd.read_csv(TRAIN_PATH, index_col=0)
test = pd.read_csv(TEST_PATH, index_col=0)
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1296675, 22)
Test shape: (555719, 22)


Train and test stay separate through the whole notebook, this is a chronological split
from the Sparkov simulator, merging them before feature engineering would let future
transactions leak into training.

In [3]:
for name, d in [('train', train), ('test', test)]:
    n_dupe_rows = d.duplicated().sum()
    n_dupe_trans = d['trans_num'].duplicated().sum()
    print(f"{name}: {n_dupe_rows} duplicate rows, {n_dupe_trans} duplicate trans_num")

train: 0 duplicate rows, 0 duplicate trans_num


test: 0 duplicate rows, 0 duplicate trans_num


## Feature engineering

In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    # standard haversine formula, all inputs in degrees
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def engineer_features(d):
    d = d.copy()
    d['trans_date_trans_time'] = pd.to_datetime(d['trans_date_trans_time'])
    d['dob'] = pd.to_datetime(d['dob'])

    d['hour'] = d['trans_date_trans_time'].dt.hour
    d['day_of_week'] = d['trans_date_trans_time'].dt.day_name()
    d['age'] = (d['trans_date_trans_time'] - d['dob']).dt.days // 365
    d['distance_km'] = haversine_km(d['lat'], d['long'], d['merch_lat'], d['merch_long'])

    drop_cols = ['cc_num', 'first', 'last', 'street', 'city', 'zip', 'trans_num', 'unix_time',
                 'dob', 'trans_date_trans_time', 'lat', 'long', 'merch_lat', 'merch_long',
                 'merchant', 'job']
    d = d.drop(columns=drop_cols)
    return d


train_fe = engineer_features(train)
test_fe = engineer_features(test)
print("Train shape after feature engineering:", train_fe.shape)
print("Test shape after feature engineering:", test_fe.shape)
train_fe.head()

Train shape after feature engineering: (1296675, 10)
Test shape after feature engineering: (555719, 10)


,category,amt,gender,state,city_pop,is_fraud,hour,day_of_week,age,distance_km
0,misc_net,4.97,F,NC,3495,0,0,Tuesday,30,78.597568
1,grocery_pos,107.23,F,WA,149,0,0,Tuesday,40,30.212176
2,entertainment,220.11,M,ID,4154,0,0,Tuesday,56,108.206083
3,gas_transport,45.00,M,MT,1939,0,0,Tuesday,52,95.673231
4,misc_pos,41.96,M,VA,99,0,0,Tuesday,32,77.556744


`merchant` (693 values) and `job` (497 values) are dropped, too high-cardinality to
encode safely at this scope without a real leakage risk, `category` (14 values) already
carries the useful part of the merchant signal (EDA showed shopping_net/misc_net at
1.3-1.6% fraud vs under 0.2% for categories like home/health_fitness).

## Categorical encoding

In [5]:
cat_cols = ['category', 'gender', 'state', 'day_of_week']

train_enc = pd.get_dummies(train_fe, columns=cat_cols)
test_enc = pd.get_dummies(test_fe, columns=cat_cols)

# align columns in case a category only appears in one split, fill missing dummies with 0
train_enc, test_enc = train_enc.align(test_enc, join='outer', axis=1, fill_value=0)

# align() sorts columns alphabetically and can separate is_fraud from the rest, put it back last
train_enc = train_enc[[c for c in train_enc.columns if c != 'is_fraud'] + ['is_fraud']]
test_enc = test_enc[[c for c in test_enc.columns if c != 'is_fraud'] + ['is_fraud']]

print("Train shape after encoding:", train_enc.shape)
print("Test shape after encoding:", test_enc.shape)
print("Columns match:", list(train_enc.columns) == list(test_enc.columns))

Train shape after encoding: (1296675, 80)
Test shape after encoding: (555719, 80)
Columns match: True


## Final checks and save

In [6]:
print("Train missing values:", train_enc.isna().sum().sum())
print("Test missing values:", test_enc.isna().sum().sum())
print("Feature count:", train_enc.shape[1] - 1)

Train missing values: 0
Test missing values: 0
Feature count: 79


In [7]:
train_enc.to_csv(TRAIN_OUT, index=False)
test_enc.to_csv(TEST_OUT, index=False)
print("Saved to", TRAIN_OUT)
print("Saved to", TEST_OUT)

Saved to ../Dataset/processed/frauddetection_train_clean.csv
Saved to ../Dataset/processed/frauddetection_test_clean.csv


## Summary

In [8]:
train_fraud_pct = train_enc['is_fraud'].value_counts(normalize=True) * 100
test_fraud_pct = test_enc['is_fraud'].value_counts(normalize=True) * 100
print(f"Train: {train_enc.shape[0]} rows, {train_enc.shape[1] - 1} features, fraud rate {train_fraud_pct[1]:.3f}%")
print(f"Test: {test_enc.shape[0]} rows, {test_enc.shape[1] - 1} features, fraud rate {test_fraud_pct[1]:.3f}%")
print(f"Missing values: {train_enc.isna().sum().sum() + test_enc.isna().sum().sum()}")

Train: 1296675 rows, 79 features, fraud rate 0.579%
Test: 555719 rows, 79 features, fraud rate 0.386%


Missing values: 0
